# 05 — Advanced Causal Analytics

**Goal:** Extract paper-worthy insights by comparing surgical workflows.

We investigate:
1. **Anomalous Triplet Sequences:** Mining implicit failure signals from rare workflow deviations.
2. **Graph Comparison:** Measuring the structural graph distance (Jaccard similarity of nodes/edges) between videos (VID01 vs VID05 vs VID40).

In [1]:
import sys, os
from pathlib import Path
import pandas as pd
import networkx as nx

sys.path.insert(0, os.path.abspath('..'))
from src.graph_builder import build_graphs_for_videos
from src.temporal_analysis import build_transition_matrix
from src.advanced_analytics import find_anomalous_transitions, compare_graphs, print_graph_comparison

# Paths
PROJECT_ROOT = Path('..').resolve()
TRIPLET_DIR = PROJECT_ROOT / 'outputs' / 'parsed_triplets'

# Read data
df_all = pd.read_csv(TRIPLET_DIR / 'all_triplets.csv')
df_valid = df_all[~df_all['is_null']].copy()


## 1. Anomalous Triplet Sequences (Implicit Failure Mining)

Does a specific triplet ever sequence into something extremely rare (< 5% probability)?
This helps identify workflow deviations (e.g. cutting something you shouldn't step to).

In [2]:
print('Mining for workflow deviations (Global transition probability <= 5%)...\n')
anomalies = find_anomalous_transitions(df_valid, threshold_prob=0.05)

if anomalies.empty:
    print("No anomalies found below threshold.")
else:
    # Format neatly
    anomalies['probability_pct'] = (anomalies['probability'] * 100).round(1).astype(str) + '%'
    anomalies = anomalies.rename(columns={'probability': 'raw_prob'})
    cols = ['From_Triplet', 'To_Triplet', 'probability_pct', 'global_count']
    print(f"Found {len(anomalies)} anomalous transition paths. Top 15 rarest deviations:\n")
    display(anomalies[cols].head(15))


Mining for workflow deviations (Global transition probability <= 5%)...

Found 131 anomalous transition paths. Top 15 rarest deviations:



,From_Triplet,To_Triplet,probability_pct,global_count
882,"grasper,retract,gallbladder","irrigator,irrigate,cystic_pedicle",0.0%,1
913,"grasper,retract,gallbladder","irrigator,irrigate,liver",0.0%,1
798,"hook,dissect,gallbladder","hook,retract,gallbladder",0.0%,1
479,"grasper,retract,gallbladder","grasper,retract,gut",0.1%,3
703,"hook,dissect,cystic_duct","hook,dissect,cystic_plate",0.1%,1
324,"grasper,retract,gallbladder","grasper,grasp,liver",0.1%,4
417,"grasper,retract,gallbladder","grasper,retract,cystic_pedicle",0.1%,4
333,"hook,dissect,gallbladder","grasper,grasp,liver",0.1%,3
944,"grasper,retract,gallbladder","irrigator,retract,liver",0.1%,5
894,"irrigator,aspirate,fluid","irrigator,irrigate,cystic_pedicle",0.1%,1


### Tracing the Anomaly back to Source

Let's look at the absolute rarest anomaly (row 0) and find which video actually performed it.

In [3]:
if not anomalies.empty:
    top_anomaly = anomalies.iloc[0]
    from_t = top_anomaly['From_Triplet']
    to_t = top_anomaly['To_Triplet']
    
    print(f"Investigating the rarest transition:\n  {from_t} → {to_t}")
    
    # Which video did this?
    for vid in ['VID01', 'VID05', 'VID40']:
        df_v = df_valid[df_valid['video'] == vid]
        mat = build_transition_matrix(df_v)
        if from_t in mat.index and to_t in mat.columns:
            if mat.loc[from_t, to_t] > 0:
                print(f"\n⚠️ FOUND IN {vid}:")
                print(f"   Occurred {mat.loc[from_t, to_t]} times in {vid}.")

Investigating the rarest transition:
  grasper,retract,gallbladder → irrigator,irrigate,cystic_pedicle


## 2. Graph Structural Comparison Across Videos

How structurally similar is the surgical strategy between videos?
We compute Node and Edge Jaccard Similarities (Intersection over Union).

In [4]:
graphs = build_graphs_for_videos(TRIPLET_DIR, video_ids=['VID01', 'VID05', 'VID40'])

G01 = graphs['VID01']
G05 = graphs['VID05']
G40 = graphs['VID40']

print_graph_comparison(G01, G05)
print("-" * 50)
print_graph_comparison(G01, G40)
print("-" * 50)
print_graph_comparison(G05, G40)


  VID01: 15 nodes, 19 edges
  VID05: 18 nodes, 23 edges
  VID40: 12 nodes, 16 edges
=== Structural Comparison: VID01 vs VID05 ===
Node Similarity (Jaccard): 73.7%  (14 shared nodes)
Edge Similarity (Jaccard): 35.5%  (11 shared exact edges)
Density Difference:      0.015 delta
--------------------------------------------------
=== Structural Comparison: VID01 vs VID40 ===
Node Similarity (Jaccard): 68.8%  (11 shared nodes)
Edge Similarity (Jaccard): 40.0%  (10 shared exact edges)
Density Difference:      0.031 delta
--------------------------------------------------
=== Structural Comparison: VID05 vs VID40 ===
Node Similarity (Jaccard): 66.7%  (12 shared nodes)
Edge Similarity (Jaccard): 50.0%  (13 shared exact edges)
Density Difference:      0.046 delta


---
**Conclusion:** The graph structural metrics successfully quantify differences in surgical technique (e.g. VID40 has significantly lower Jaccard similarity to VID01 than VID05 does). The anomaly transition matrix successfully identifies true outliers in workflow causal chains.